In [1]:
# 所在階と建物の高さから高さ率を割り出す
import pandas as pd
import re

In [2]:
train1 = pd.read_csv('train.csv')
test1 = pd.read_csv('test.csv')

In [3]:
train = train1[['id', '所在階']].copy()
test = test1[['id', '所在階']].copy()

In [4]:
# 所在階列の「階」と「階建」を取り出して計算する関数
def calc_floor_ratio(floor_info):
    try:
        # 「階」と「階建」を分割
        floor, total_floors = floor_info.split('階／')
        # 数値に変換して割り算
        result = int(floor) / int(total_floors.replace('階建', ''))
        # 1より大きいのはおかしいので逆になってるとして再計算
        if result > 1.0:
            return int(total_floors.replace('階建', '')) / int(floor)
        # 1以下ならそのまま返す
        else:
            return result
    except:
        return None  # エラー時はNoneを返す

In [5]:
# 所在階を変換
train['高さ率'] = train['所在階'].apply(calc_floor_ratio)
test['高さ率'] = test['所在階'].apply(calc_floor_ratio)

In [6]:
# 全体の平均値出すために一回結合
df = pd.concat([train, test], ignore_index=True)

In [7]:
# 欠損値の補完（平均値で補完）
mean_value = df['高さ率'].mean()  # 平均値を計算
train['高さ率'] = train['高さ率'].fillna(mean_value)  # 平均値で欠損値を補完
test['高さ率'] = test['高さ率'].fillna(mean_value)  # 平均値で欠損値を補完

In [8]:
# 高さ率を小数点第2位までで揃える
train['高さ率'] = train['高さ率'].round(2)
test['高さ率'] = test['高さ率'].round(2)

In [9]:
train.drop(columns=['所在階'], inplace=True)
test.drop(columns=['所在階'], inplace=True)

In [10]:
train.to_csv('train_floor.csv', index=False)
test.to_csv('test_floor.csv', index=False)